# Week 3 · Day 2: Understand sigmoid probabilities

**Course:** Lab 02: Sigmoid

The teaching question is **“Is this source's absolute G magnitude below 4?”** Class 1 means yes. This is a brightness label, not a verified stellar type. Only `bp_rp` (color) enters the model; magnitude and parallax are used to construct the answer and must stay out of the inputs.

Day 1 saves a source-disjoint 60/20/20 split. Days 2–6 explore training and validation; Day 7 evaluates the reserved test set. Absolute magnitudes are uncorrected for dust, and the archive sample may not represent all Gaia sources.

## Setup

[Prerequisites and setup explanation](week3_prerequisites.ipynb). Run cells from top to bottom. Run Day 1 first. This notebook reads its saved data.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)


## Load the Day 1 data

These lines select the saved splits. `X` contains color and `y` contains the brightness label.

In [ ]:
prepared_path = OUTPUT / 'learning_data.csv'
if not prepared_path.is_file():
    raise FileNotFoundError('Run Week 3 Day 1 first to create learning_data.csv.')

# Keep source IDs as text so their digits are not rounded.
gaia = pd.read_csv(prepared_path, dtype={'source_id': 'string'})
train = gaia.loc[gaia['split'] == 'train'].copy()
validation = gaia.loc[gaia['split'] == 'validation'].copy()
test = gaia.loc[gaia['split'] == 'test'].copy()
feature_columns = ['bp_rp']
X_train = train[feature_columns]
y_train = train['is_bright']
X_val = validation[feature_columns]
y_val = validation['is_bright']
print('Training rows:', len(train), '| Validation rows:', len(validation))


## 1. Define sigmoid

The formula is p = 1 / (1 + exp(−z)). At z=0, p=0.5. The equivalent NumPy form below avoids overflow for extreme scores.

In [ ]:
def sigmoid(z):
    """Turn a score into a probability, safely even for very large scores."""
    # This equals 1 / (1 + exp(-z)); logaddexp avoids overflow.
    return np.exp(-np.logaddexp(0, -np.asarray(z, dtype=float)))


## 2. Try some scores

Expected outputs: probability table and S-shaped curve. Change the scores and predict the outputs before running.

In [ ]:
scores = np.array([-1000, -2, 0, 2, 1000])
display(pd.DataFrame({'score': scores, 'probability': sigmoid(scores)}))
z = np.linspace(-10, 10, 300)
visualize.plot_binary_curves(z, {'sigmoid': sigmoid(z)}, xlabel='Score z', ylabel='Probability', title='Sigmoid', threshold=0.5)

## 3. Scale color using training data

W2 used feature scaling too. Fit the scaler on training rows and apply those same values to validation rows.

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
print('Training mean color:', scaler.mean_[0])

## 4. Build a prediction by hand

These weights are deliberately hand-picked; they are not a trained Gaia model. Follow color → score → probability → label. Expected output: all four stages for ten rows. Change w or b to see how predictions move.

In [ ]:
w = np.array([-1.0])
b = 0.0
scores = X_val_scaled @ w + b
probabilities = sigmoid(scores)
predictions = (probabilities >= 0.5).astype(int)
preview = validation[['source_id', 'bp_rp', 'is_bright']].head(10).copy()
preview['score'] = scores[:10]
preview['probability'] = probabilities[:10]
preview['prediction'] = predictions[:10]
display(preview)

## Reflection questions and answers

1. **What is sigmoid(0)?** The formula is `1 / (1 + exp(-z))`. At `z = 0`, `exp(0) = 1`, so the answer is `1 / (1 + 1) = 0.5`. A score of zero is the midpoint: the model assigns equal probability to labels 0 and 1.

2. **Have we learned weights yet?** No. In this notebook we set `w = [-1.0]` and `b = 0.0` ourselves to see how a color becomes a score, then a probability. Those values did not come from the Gaia labels. In Day 3, `model.fit(X_train, y_train)` learns weights from the training data.

3. **Does a probability guarantee a correct label?** No. A probability is the model's estimate. For example, a prediction of 0.8 for class 1 can still be wrong for an individual star. We check performance on stars the model did not train on. We can also check whether predictions near 0.8 are correct about 80% of the time; that is called **calibration**.


## References

Course labs: `../ML-coursework/W3/Optional Labs/` in the workspace.

[ESA Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) · [Extinction guidance](https://www.cosmos.esa.int/web/gaia/edr3-extinction-law) · [Scikit-learn logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)